# 01 · Dataset + classical ML baselines (Random Forest, XGBoost)

**Goal:** build the agent-routing dataset, look at it, and train the two classical baselines.

Each ticket has free text (`query`) plus tabular metadata (`account_tier`, `channel`, `urgency_score`,
`past_failures`). The model must output a **typed decision**:

| field | type | values |
|---|---|---|
| `route_target` | choice | billing, orders, account, tech_support, fraud, general |
| `priority_level` | ordinal | low < medium < high < critical |
| `requires_human` | yes/no | True / False |

Runs fine on a CPU runtime; no GPU needed for this notebook.

## Setup

Works both on **Colab** and locally. On Colab this clones the repo and moves into it;
locally (opened from `notebooks/`) it just moves up to the repo root so `import src...` works.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/AdityaThakare72/typed-decision-models.git"
BRANCH = "main"

if not os.path.exists("src/dataset.py"):
    if os.path.exists("../src/dataset.py"):      # running locally from notebooks/
        os.chdir("..")
    else:                                         # fresh Colab runtime
        if not os.path.exists("typed-decision-models"):
            subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL], check=True)
        os.chdir("typed-decision-models")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

Install the light dependencies (Colab already has most of them, so this is quick).

In [ ]:
%pip install -q scikit-learn xgboost pandas matplotlib

## 1. Build the dataset

Two options:
- `USE_HF = False` → the **offline synthetic** builder: templated queries, deterministic (same seed → same data).
- `USE_HF = True` → **real customer-support queries** from Hugging Face (Bitext), with the same metadata + labelling rules on top.
  Note: Bitext has no fraud tickets, so the `fraud` route will be empty.

Labels come from simple rules (see `decide_priority` / `decide_requires_human` in `src/dataset.py`) plus 5% random label
noise, so no model can be perfect. The splits are saved to `data/` so every notebook uses the same rows.

In [ ]:
from src.dataset import (build_synthetic_dataset, load_hf_dataset, train_val_test_split,
                         save_splits, LABEL_COLUMNS, FEATURE_COLUMNS)

USE_HF = False
N_ROWS = 3000
SEED = 42

df = load_hf_dataset(N_ROWS, SEED) if USE_HF else build_synthetic_dataset(N_ROWS, SEED)
train_df, val_df, test_df = train_val_test_split(df, seed=SEED)
save_splits(train_df, val_df, test_df, "data")
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
df.head(10)

## 2. Quick EDA

First, how balanced are the three labels? Routes are roughly even by design. Priority is skewed
(few `critical` tickets) — that is why we report **macro-F1**, which weighs every class equally.

In [ ]:
for column in LABEL_COLUMNS:
    print(df[column].value_counts(normalize=True).round(3).to_dict())

How do the labels depend on the metadata? Each row below shows, per account tier, the share of tickets in each
priority level. Enterprise tickets lean towards `high`/`critical`, which is exactly the kind of signal the tabular
features must carry (the text alone can't tell you the tier).

In [ ]:
import pandas as pd

priority_order = ["low", "medium", "high", "critical"]
(pd.crosstab(df["account_tier"], df["priority_level"], normalize="index")[priority_order]
   .round(2))

Same idea for the yes/no field: share of tickets that need a human, by route and by number of past failures.

In [ ]:
print(df.groupby("route_target")["requires_human"].mean().round(2).sort_values())
print()
print(df.groupby("past_failures")["requires_human"].mean().round(2))

Query lengths: they are short (tickets, not essays), which matters later: TF-IDF works well on short text,
and transformer latency grows with input length.

In [ ]:
lengths = df["query"].str.split().str.len()
lengths.describe().round(1)

## 3. Train Random Forest and XGBoost

Both use the same feature pipeline (`build_preprocessor` in `src/models/classical_ml.py`):
- `query` → **TF-IDF** on words + word pairs
- `account_tier`, `channel` → **one-hot**
- `urgency_score`, `past_failures` → **standardised** (mean 0, std 1)

Trees predict one label at a time, so each model trains **three classifiers** (one per output field).

In [ ]:
from src.models.classical_ml import ClassicalDecisionModel

rf = ClassicalDecisionModel("rf", seed=SEED).fit(train_df)
xgb = ClassicalDecisionModel("xgb", seed=SEED).fit(train_df)
print("trained:", rf.name, xgb.name)

## 4. Benchmark

`run_benchmark` (in `src/interface.py`) runs every model on the same test set and reports:
- accuracy + macro-F1 per field, plus **exact_match** (all three fields right at once, what a router really needs)
- latency **p50 / p95** over 100 single-ticket predictions (after a short warm-up)
- model size in MB (pickled pipelines)

In [ ]:
from src.interface import run_benchmark

results = run_benchmark([rf, xgb], test_df, n_latency_runs=100)
results.round(3)

Save the results so notebook 03 (or the README) can reuse them without retraining.

In [ ]:
os.makedirs("artifacts", exist_ok=True)
results.to_csv("artifacts/results_classical.csv")
print(open("artifacts/results_classical.csv").read())

## 5. What did XGBoost learn?

Top features for the `priority_level` classifier. Expect the numeric metadata (urgency, past failures) and the tier
one-hots near the top, with urgent words ("asap", "urgent") from TF-IDF after them.

In [ ]:
from src.models.classical_ml import feature_names

clf = xgb.pipelines["priority_level"].named_steps["clf"]
importance = pd.Series(clf.feature_importances_, index=feature_names(xgb, "priority_level"))
importance.sort_values(ascending=False).head(15).round(4)

## 6. Error check: where does XGBoost get priority wrong?

A confusion table for priority. Most mistakes should be between **neighbouring** levels (medium↔high);
jumping low↔critical would be a worse error for a router.

In [ ]:
from src.interface import outputs_to_frame

pred = outputs_to_frame(xgb.predict_batch(test_df))
pd.crosstab(test_df["priority_level"], pred["priority_level"],
            rownames=["true"], colnames=["predicted"]).reindex(index=priority_order, columns=priority_order, fill_value=0)

**Takeaways to fill in after running:**
- Route accuracy: _TODO_
- Priority macro-F1 (the hard field): _TODO_
- Latency p50/p95 on CPU: _TODO_

Next: `02_laya_base_vs_finetuned.ipynb`.